# Step 1a: Canary-1B-v2 transcribes the recording (GPU)

Kernel: **NeMo ASR** (`~/nemoenv`, see `../../environments/README.md`). Run `../../environments/patch_nemo_py310.py` once before the first run, otherwise a whole meeting fails with `SyntaxError: invalid syntax`.

Set `MEETING` and `AUDIO` in the first line, then run the cell. Paths are relative to `pipeline/`; the cell changes to that folder. It converts the recording to 16 kHz mono (`outputs/<MEETING>/<audio name>_16k_mono.wav`), transcribes it with word timestamps and saves the words to `outputs/<MEETING>/_words_step1a.json`. Then run `step1b_pyannote_speakers.ipynb` with the same two values.

In [ ]:
MEETING, AUDIO = "april27", "data/Team-Meeting_2026-04-27.flac"

import os, sys, json
from pathlib import Path
from dataclasses import asdict
ROOT = Path.cwd().parent if Path.cwd().name == "cluster" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, os.getcwd())
from thesis_pipeline.canary_diarize import ensure_16k_mono, load_canary, transcribe_words

out = Path("outputs") / MEETING; out.mkdir(parents=True, exist_ok=True)
wav = ensure_16k_mono(AUDIO, tmp_dir=str(out))
model = load_canary()
words = transcribe_words(model, wav)
(out / "_words_step1a.json").write_text(json.dumps([asdict(w) for w in words], ensure_ascii=False), encoding="utf-8")
print(len(words), "words ->", wav)